In [31]:
import os
import re
import json
import fitz 
import asyncio
import pandas as pd
from glob import glob
from tqdm import tqdm
from pathlib import Path
from tqdm import tqdm
from tqdm.asyncio import tqdm_asyncio
from mistralai import Mistral
from dotenv import load_dotenv
from types import SimpleNamespace

from openai import OpenAI, AsyncOpenAI

load_dotenv()
api_key = os.environ["MISTRAL_API_KEY"]

client = Mistral(api_key=api_key)
gpt_client = OpenAI()

In [32]:
def process_filename(file_path: str | Path, limit_folder=None) -> str:
    path_obj = Path(file_path).resolve()
    return f"\\\\?\\{path_obj}"

def upload_pdf_mistral(filepath: str):
    safe_path = process_filename(filepath)
    
    with open(safe_path, "rb") as f:
        uploaded_pdf = client.files.upload(
            file={
                "file_name": Path(filepath).name,
                "content": f,
            },
            purpose="ocr"
        )
    
    signed_url = client.files.get_signed_url(file_id=uploaded_pdf.id)
    return signed_url.url

def parse_pdf_text(filepath: str):
    safe_path = process_filename(filepath)
    
    try:
        with open(safe_path, "rb") as f:
            file_bytes = f.read()
            
        doc = fitz.open(stream=file_bytes, filetype="pdf")
        
        pages_list = []
        for page in doc:
            text_content = page.get_text("text") 
            
            page_obj = SimpleNamespace(markdown=text_content)
            pages_list.append(page_obj)
        
        return SimpleNamespace(pages=pages_list)
        
    except OSError as e:
        print(f"Error opening {Path(filepath).name}: {e}")
        return None

def parse_pdf_ocr(filepath: str):
    try:
        ocr_response = client.ocr.process(
            model="mistral-ocr-latest",
            document={
                "type": "document_url",
                "document_url": upload_pdf_mistral(filepath=filepath),
            },
            include_image_base64=True,
        )
        return ocr_response
    except Exception as e:
        print(f"Error OCR-ing {filepath}: {e}")
        return None

def parse_pdf(filepath: str, method: str = "ocr"):
    if method == "ocr":
        return parse_pdf_ocr(filepath)
    elif method == "text":
        return parse_pdf_text(filepath)
    else:
        raise ValueError("Method must be 'ocr' or 'text'")


In [8]:
sample_judgement_path = r"C:\Users\ghana\OneDrive\kuliah\Semester 8\TA\experiment\downloads\Putusan_PTA_SURABAYA_Nomor_0059_Pdt.G_2016_PTA.Sby..pdf"
responses = parse_pdf(sample_judgement_path, method="text")
print(responses.pages[-1].markdown)

hkama
ahkamah Agung Repub
ahkamah Agung Republik Indonesia
mah Agung Republik Indonesia
blik Indonesi
Direktori Putusan Mahkamah Agung Republik Indonesia
putusan.mahkamahagung.go.id
Drs. H. Hamberi Hadi, S.H., M.H.             H. Masruri Syuhadak, S.H., M.H. 
Panitera Pengganti,
Ttd.
Dra. Hj. Suffana Qomah
Rincian Biaya Proses:
- Pemberkasan ATK: Rp. 139.000,-
- Redaksi
: Rp.     5.000,-
- Meterai
: Rp.     6.000,-
Jumlah
: Rp. 150.000,-
(seratus lima puluh ribu rupiah)
UNTUK SALINAN 
PENGADILAN TINGGI AGAMA SURABAYA
PANITERA,
      H. MUH. IBRAHIM, S.H., M.M.
.
Disclaimer
Kepaniteraan Mahkamah Agung Republik Indonesia berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen Mahkamah Agung untuk pelayanan publik, transparansi dan akuntabilitas
pelaksanaan fungsi peradilan. Namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang kami sajikan, hal mana akan terus kami perbaiki dar

In [ ]:
judgement_summarization_prompt = """
You are a legal data assistant specializing in Indonesian Court Decisions. 
Process the provided judgment text and output a JSON object containing exactly two fields:

1. "incidents": 
Extract legal incidents from a given judgment to understand why specific sections or articles of law were
cited. These extracted incidents will later be matched with relevant sections and articles.
Please process the given legal judgment and focus on the following instructions:
Objective: Identify and extract all legal incidents referenced in the judgment, focusing on the key facts
and legal issues of the case.
Structure: Phrase each incident concisely and neutrally. Exclude case-specific details (e.g., names, dates,
case numbers). The extracted incidents should be rich in legal reasoning and sufficiently descriptive to
enable accurate section/article matching.
Focus Areas: Capture the core facts and issues underlying the case.
Language: Bahasa Indonesia.

2. "relevant_laws":
Objective: Extract the specific Articles (Pasal) and Regulations (UU, KUHP, etc.) used by the judge as the legal basis for the decision.
Format: A list of strings. Each string should contain the specific article and regulation (e.g., "Pasal 1365 KUHPerdata").
Constraint: Only include laws explicitly cited in the judge's consideration or decision part.
Output: [REGULATION_NAME] Pasal [ARTICLE_NUMBER] -> [KUHAPerdata] Pasal 100

Output must be valid JSON only. Do not add markdown formatting (```json).
"""

full_judgment_text = "\n\n".join([page.markdown for page in responses.pages])

completion = gpt_client.chat.completions.create(
    model="gpt-5-mini", 
    messages=[
        {
            "role": "user", 
            "content": f"{judgement_summarization_prompt}\n\nJudgement Text:\n\n{full_judgment_text}"
        }
    ],
    response_format={"type": "json_object"}
)

In [23]:
json.loads(completion.choices[0].message.content)

{'incidents': ['Penggugat menuntut agar dua bidang tanah ditetapkan sebagai harta bersama hasil perkawinan; tergugat membantah dengan dalil bahwa tanah tersebut adalah harta warisan dari orang tua pewaris.',
  'Perselisihan mengenai ada tidaknya ahli waris serta penentuan dan pembagian bagian waris yang timbul akibat kematian pewaris.',
  'Sengketa tentang asal kepemilikan tanah termasuk klaim adanya praktik jog-jogan/nyusuki (kompensasi kepada ahli waris lain) yang, jika terbukti, mempengaruhi klasifikasi harta sebagai warisan atau harta bersama.',
  'Persoalan pembuktian karena gugatan dibantah: pihak penggugat dan tergugat masing-masing harus memenuhi beban pembuktian sesuai ketentuan hukum acara perdata/HI R.',
  'Eksepsi tergugat terkait obscuur libel (ketidakjelasan gugatan) yang mencakup perubahan gugatan, ketidaksesuaian identitas pihak, tidak lengkapnya pihak yang dianggap berkepentingan, dan ketidakjelasan jenis tanah; pengadilan menolak eksepsi tersebut.',
  'Putusan pengadi

In [34]:
gpt_client = AsyncOpenAI()
LLM_MODEL = "gpt-5-mini"    
BATCH_SIZE = 10
sem = asyncio.Semaphore(BATCH_SIZE)
output_path = r"C:\Users\ghana\OneDrive\kuliah\Semester 8\TA\data\judgement\judgement_to_content.json"

async def process_single_judgement(filepath):
    async with sem:  
        try:
            responses = await asyncio.to_thread(parse_pdf, filepath)
            
            if not responses:
                return filepath, None
            
            full_judgment_text = "\n\n".join([page.markdown for page in responses.pages])

            completion = await gpt_client.chat.completions.create(
                model=LLM_MODEL, 
                messages=[
                    {
                        "role": "user", 
                        "content": f"{judgement_summarization_prompt}\n\nJudgement Text:\n\n{full_judgment_text}"
                    }
                ],
                response_format={"type": "json_object"}
            )
            raw_content = completion.choices[0].message.content
            
            try:
                content = json.loads(raw_content)
            except json.JSONDecodeError:
                print(f"Got invalid json on {filepath}, calling llm to fix")
                fix_completion = await gpt_client.chat.completions.create(
                    model=LLM_MODEL, 
                    messages=[
                        {
                            "role": "user", 
                            "content": f"Fix the malformed json below into valid json:\n\n{raw_content}"
                        }
                    ],
                    response_format={"type": "json_object"}
                )
                content = json.loads(fix_completion.choices[0].message.content)
            
            return filepath, content

        except Exception as e:
            print(f"Error processing {filepath}: {e}")
            return filepath, None

judgement_filepaths = glob("downloads/*.pdf")
judgement_to_content = {}

try:
    with open(output_path, "r") as f:
        judgement_to_content = json.load(f)
except FileNotFoundError:
    pass

files_to_process = [f for f in judgement_filepaths if f not in judgement_to_content]
tasks = [process_single_judgement(f) for f in files_to_process]

if tasks:
    for future in tqdm_asyncio.as_completed(tasks, total=len(tasks)):
        filepath, content = await future
        if content:
            judgement_to_content[filepath] = content

    with open(output_path, "w") as f:
        json.dump(judgement_to_content, f, indent=2)
else:
    print("No new files to process.")

100%|██████████| 85/85 [07:12<00:00,  5.09s/it]


In [29]:
with open(r"C:\Users\ghana\OneDrive\kuliah\Semester 8\TA\data\judgement\judgement_to_content.json", "w") as f:
    json.dump(judgement_to_content, f)